# Example: Bootstrap Uncertainty in a Single Index Model
This example estimates a SIM from the frozen course dataset, compares empirical-residual and Gaussian parametric bootstraps, and checks whether the residuals retain the heavy tails or volatility memory introduced in L3a.

> __Learning Objectives:__
>
> * __Estimate a SIM with an explicit unit convention:__ Fit $(\alpha,\beta,\sigma_{g,\varepsilon})$ to annualized growth-rate observations.
> * __Quantify sampling uncertainty:__ Construct empirical percentile intervals and compare bootstrap standard errors with the analytical ridge-sandwich result.
> * __Separate two uncertainty models:__ Explain what residual and parametric bootstraps preserve and discard.
> * __Audit residual adequacy:__ Use tail and ACF diagnostics to decide whether i.i.d. bootstrap draws are sufficient for the downstream question.

___


## Setup, Data, and Prerequisites
The reusable estimator and bootstrap live in the course package. A local random-number generator is used inside each bootstrap, so running this notebook does not mutate Julia's global RNG state.


In [ ]:
include(joinpath(@__DIR__, "Include.jl"));


In [ ]:
raw_dataset = MyTrainingMarketDataSet()["dataset"];
maximum_days = nrow(raw_dataset["AAPL"]);
dataset = Dict(ticker => frame for (ticker, frame) in raw_dataset if nrow(frame) == maximum_days);
tickers = sort(collect(keys(dataset)));
Δt = 1 / 252;
growth_rates = log_growth_matrix(dataset, tickers; Δt=Δt, risk_free_rate=0.0);

market_ticker = "SPY";
asset_ticker = "QQQ";
market = growth_rates[:, findfirst(==(market_ticker), tickers)];
asset = growth_rates[:, findfirst(==(asset_ticker), tickers)];
println("Fitting $(asset_ticker) on $(market_ticker) with $(length(market)) observations")


## Task 1: Fit and Interpret the SIM
With $g_t=\Delta t^{-1}\log(P_t/P_{t-1})$, the fitted residual standard deviation $\sigma_{g,\varepsilon}$ uses the same growth-rate observation convention. The practitioner-style annual diffusion volatility is $\sqrt{\Delta t}\,\sigma_{g,\varepsilon}$.

$$
g_{i,t}=\alpha_i+\beta_i g_{m,t}+\varepsilon_{i,t}.
$$


In [ ]:
estimate = estimate_sim(market, asset, asset_ticker; δ=0.0);
parameter_table = DataFrame(
    parameter=["alpha", "beta", "residual growth-rate std", "residual diffusion vol", "R-squared"],
    estimate=[estimate.α, estimate.β, estimate.σ_ε, sqrt(Δt) * estimate.σ_ε, estimate.r²],
);
pretty_table(parameter_table; table_format=TextTableFormat(borders=text_table_borders__simple))

fitted = estimate.α .+ estimate.β .* market;
residuals = asset .- fitted;
scatter(fitted, asset; ms=2, alpha=0.35, c=:navy, label="observations",
    xlabel="SIM-predicted growth rate", ylabel="Observed growth rate", framestyle=:box);
lims = extrema(vcat(fitted, asset));
plot!([lims...], [lims...]; c=:red, ls=:dash, label="x=y")


## Task 2: Compare Residual and Parametric Bootstraps
The residual bootstrap samples centered fitted residuals with replacement, retaining the empirical one-day marginal distribution. The parametric bootstrap replaces that marginal with a fitted Gaussian. Both methods hold the observed market design vector fixed, and both treat innovations as independent across time.

The reported intervals are empirical percentile intervals. For ridge regression, the analytical covariance is the sandwich
$$
\widehat{\operatorname{Cov}}(\hat\theta)=s_\varepsilon^2(\mathbf X^\top\mathbf X+\delta\mathbf I)^{-1}\mathbf X^\top\mathbf X(\mathbf X^\top\mathbf X+\delta\mathbf I)^{-1},
$$
which reduces to $s_\varepsilon^2(\mathbf X^\top\mathbf X)^{-1}$ only when $\delta=0$.


In [ ]:
residual_bootstrap = bootstrap_sim(market, asset, asset_ticker;
    n_bootstrap=1000, seed=5660, method=:residual);
parametric_bootstrap = bootstrap_sim(market, asset, asset_ticker;
    n_bootstrap=1000, seed=5660, method=:parametric);

bootstrap_table = DataFrame(
    method=["residual", "parametric"],
    beta_point=fill(estimate.β, 2),
    beta_ci_low=[residual_bootstrap.confidence_intervals.beta[1], parametric_bootstrap.confidence_intervals.beta[1]],
    beta_ci_high=[residual_bootstrap.confidence_intervals.beta[2], parametric_bootstrap.confidence_intervals.beta[2]],
    beta_bootstrap_se=[residual_bootstrap.bootstrap_standard_errors.beta, parametric_bootstrap.bootstrap_standard_errors.beta],
    beta_theory_se=[residual_bootstrap.theoretical_standard_errors.beta, parametric_bootstrap.theoretical_standard_errors.beta],
    residual_std_ci_low=[residual_bootstrap.confidence_intervals.sigma_epsilon[1], parametric_bootstrap.confidence_intervals.sigma_epsilon[1]],
    residual_std_ci_high=[residual_bootstrap.confidence_intervals.sigma_epsilon[2], parametric_bootstrap.confidence_intervals.sigma_epsilon[2]],
);
pretty_table(bootstrap_table; table_format=TextTableFormat(borders=text_table_borders__simple))


In [ ]:
p1 = histogram(residual_bootstrap.beta_samples; bins=40, normalize=:pdf,
    c=:navy, alpha=0.65, label="residual", xlabel="Bootstrap beta", ylabel="Density");
histogram!(p1, parametric_bootstrap.beta_samples; bins=40, normalize=:pdf,
    c=:orange, alpha=0.45, label="parametric");
vline!(p1, [estimate.β]; c=:red, lw=2, ls=:dash, label="point estimate");
p2 = histogram(residual_bootstrap.sigma_epsilon_samples; bins=40, normalize=:pdf,
    c=:purple, alpha=0.65, label="residual", xlabel="Residual growth-rate std", ylabel="Density");
histogram!(p2, parametric_bootstrap.sigma_epsilon_samples; bins=40, normalize=:pdf,
    c=:green4, alpha=0.45, label="parametric");
plot(p1, p2; layout=(1, 2), size=(1000, 400))


## Task 3: Audit the Residual Model
A bootstrap is only as credible as its innovation model. We therefore carry the L3a diagnostics into the SIM residuals. Heavy tails favor empirical residual sampling for one-step marginal uncertainty. Significant ACF in $|\varepsilon_t|$ warns that either i.i.d. bootstrap destroys volatility persistence; a block bootstrap or a regime/time-varying model is then needed for multi-day path risk.


In [ ]:
lags = [1, 5, 10, 20, 50];
residual_report = stylized_facts_report(residuals; lags=lags);
residual_diagnostics = DataFrame(
    lag=lags,
    residual_acf=residual_report.raw_acf,
    absolute_residual_acf=residual_report.absolute_acf,
    white_noise_95=fill(residual_report.white_noise_95_band, length(lags)),
);
pretty_table(residual_diagnostics; table_format=TextTableFormat(borders=text_table_borders__simple));
println("Absolute-residual Hill tail index: $(round(residual_report.tail_index, digits=2))")


## Summary

> __Key Takeaways:__
>
> * __Point estimates are not decision distributions:__ Bootstrap draws expose the sampling uncertainty hidden by one $(\hat\alpha,\hat\beta,\hat\sigma_{g,\varepsilon})$ tuple.
> * __Residual and parametric bootstraps answer different model-risk questions:__ Their difference is evidence about sensitivity to the Gaussian innovation assumption.
> * __The ridge covariance is a sandwich:__ Dropping the middle $\mathbf X^\top\mathbf X$ factor is valid only in the unregularized OLS case.
> * __L3 diagnostics remain relevant:__ Marginal heavy tails and volatility clustering determine whether an i.i.d. bootstrap is adequate for the intended horizon.

The companion L6b forward-validation notebook carries these bootstrap draws into portfolio variance, allocation distance, and optimization regret.
___

## Disclaimer and Risks
This material is for educational purposes only and does not constitute investment advice. Bootstrap intervals quantify uncertainty under the fitted design and resampling assumptions; they do not cover structural breaks, omitted factors, or future regime changes.
